# Customer Metrics Correlation & Pairwise Analysis
This notebook demonstrates how to load/generate customer data, compute Pearson correlation coefficients between numeric features, analyze relationship strengths, and visualize them using heatmaps and pairwise scatter plots.

## Objectives
1. Compute Pearson correlations between numeric features.
2. Visualize the correlation matrix as a masked heatmap with annotations.
3. Display pairwise relationships / scatter matrices for key variable pairs.
4. Summarize the strongest positive and negative relationships.

### Step 1: Imports and Configuration
We import our data generation and analysis modules from the `src` package.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the parent directory to the system path so we can import 'src'
sys.path.append(os.path.abspath('..'))

from src.generator import generate_synthetic_data
from src.analyzer import (
    load_and_preprocess_data,
    compute_pearson_correlation,
    get_strongest_relationships,
    plot_correlation_heatmap,
    plot_pairwise_relationships
)

# Set inline plotting
%matplotlib inline
sns.set_theme(style="ticks")
print("Libraries successfully imported!")

### Step 2: Load / Generate Dataset
We check if the dataset exists in the `data/` folder, and if not, we generate a synthetic customer metrics dataset containing 500 samples.

In [ ]:
data_path = '../data/sample_data.csv'

if not os.path.exists(data_path):
    print("Generating sample dataset...")
    df = generate_synthetic_data(num_samples=500, random_seed=42)
    os.makedirs('../data', exist_ok=True)
    df.to_csv(data_path, index=False)
else:
    print("Loading existing dataset...")
    df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
df.head()

### Step 3: Compute Pearson Correlation
Pearson's correlation coefficient ($r$) measures the linear relationship between two variables, ranging from -1 (perfect negative) to +1 (perfect positive).

In [ ]:
# Preprocess to select only numeric features
_, numeric_df = load_and_preprocess_data(data_path)

# Compute correlation matrix
corr_matrix = compute_pearson_correlation(numeric_df)
corr_matrix.round(3)

### Step 4: Summarize Strongest Relationships
We extract the top 3 strongest positive and negative correlations (ignoring self-correlations and duplicate pairs).

In [ ]:
relationships = get_strongest_relationships(corr_matrix, top_n=3)

print("=== Strongest Positive Relationships ===")
for idx, row in relationships['positive'].iterrows():
    print(f"{idx+1}. {row['Feature_1']} and {row['Feature_2']}: {row['Correlation']:.3f}")

print("\n=== Strongest Negative Relationships ===")
for idx, row in relationships['negative'].iterrows():
    print(f"{idx+1}. {row['Feature_1']} and {row['Feature_2']}: {row['Correlation']:.3f}")

### Step 5: Visualize as a Masked Heatmap
To make the heatmap clean and readable:
- We mask the upper triangle because the matrix is symmetric.
- We annotate each cell with the correlation value.
- We use a custom blue-to-red diverging colormap to distinguish positive/negative relationships easily.

In [ ]:
fig, ax = plot_correlation_heatmap(corr_matrix, output_path='../output/correlation_heatmap.png', theme='light')
plt.show()

### Step 6: Pairplots / Scatter Matrix for Key Variables
We use pairplots to visualize the distributions of individual variables and the scatter relationships between key pairs, color-coded by the customer's gender.

In [ ]:
key_vars = ['Age', 'Annual_Income', 'Spending_Score', 'Savings']
g = plot_pairwise_relationships(df, columns=key_vars, output_path='../output/pairwise_relationships.png', hue_column='Gender', theme='light')
plt.show()